# Stage 4 — Portfolio Construction Comparison

Does a smarter *within-leg* weighting scheme beat inverse-vol **net of costs**?

We hold everything else fixed — the combined **ALL** quintile carry sort, monthly rebalance, the
same 1M-forward cost model — and vary only how names are weighted inside each leg:
**equal**, **inverse-vol** (current), **equal-risk-contribution (ERC)**, and **mean-variance
(MVO)** with μ = observable forward-implied carry (no return forecasting). Every variant is
re-vol-targeted to 10% annualised so the Sharpe comparison is scale-free; the differentiators
then become net-of-cost efficiency and tail shape. The honest prior (plan §10): optimization
beats inverse-vol *modestly at best*.

New helpers live in `fx_utils`: `shrunk_cov` (Ledoit–Wolf), `erc_weights` (cyclical coordinate
descent), `mvo_weights` (SLSQP max-Sharpe under leg-gross + single-name-cap). `carry_portfolio`
gained a `weighting` kwarg (`equal | inv_vol | erc | mvo`); the default reproduces the earlier
inverse-vol behaviour bit-for-bit.

In [1]:
# ===== 0. Setup =====
from pathlib import Path
import numpy as np, pandas as pd
import fx_utils as fx

OUT = Path('outputs')
g10_px = fx.load_wide('g10_fx_spot_forward')
em_px  = fx.load_wide('em_fx_spot_forward')
spots  = fx.spots_usd_per_fx(g10_px, em_px)
carry  = fx.carry_panel(g10_px, em_px, tenor='1M')
xret   = fx.excess_returns(spots, carry)

UNIVERSE_ALL = [c for c in xret.columns if c not in ('HKD','DKK','CNY')]   # ~27 names
N_BUCKETS    = 5                                                            # quintile sort
BENCH        = 'FXCTEM8'
SCHEMES      = ['equal', 'inv_vol', 'erc', 'mvo']
VOL_TARGET   = 0.10
WINDOW       = slice('2007-05-01', '2026-06-30')

bmk = fx.benchmark_returns()
hs_out, hs_pts = fx.forward_halfspreads('1M')
print('xret', xret.shape, '| ALL universe', len(UNIVERSE_ALL), '| benchmark', BENCH)

xret (5094, 29) | ALL universe 27 | benchmark FXCTEM8


## 1. Weight panels — the four within-leg schemes

One unit book per scheme (combined quintile sort, gross 2 / net 0), then vol-targeted to 10%.
Everything but `weighting=` is identical, so any difference is purely the weighting scheme.

In [2]:
# ===== 1. Unit + vol-targeted weight panels =====
w_unit, w = {}, {}
for s in SCHEMES:
    w_unit[s] = fx.carry_portfolio(carry, xret, n_buckets=N_BUCKETS, universe=UNIVERSE_ALL,
                                   weighting=s)
    w[s]      = fx.vol_target_weights(w_unit[s], xret, target=VOL_TARGET)
print('weight panels:', SCHEMES)
print('avg gross leverage (vol-targeted, last obs):',
      {s: round(float(w[s].abs().sum(axis=1).dropna().iloc[-1]), 2) for s in SCHEMES})

weight panels: ['equal', 'inv_vol', 'erc', 'mvo']
avg gross leverage (vol-targeted, last obs): {'equal': 4.94, 'inv_vol': 6.61, 'erc': 6.51, 'mvo': 5.63}


## 2. Tracks — gross and net of costs

Gross = Σ w·r; net = gross − round-trip cost (new notional pays the outright half-spread,
maintained notional rolls at the points half-spread) — the same cost model as every other stage.

In [3]:
# ===== 2. Gross / net tracks =====
tracks = {}
for s in SCHEMES:
    g = fx.portfolio_returns(w[s], xret).rename(f'ALL_{s}_gross')
    c = fx.roundtrip_cost(w[s], hs_out, hs_pts)
    n = (g - c).rename(f'ALL_{s}_net')
    tracks[g.name] = g
    tracks[n.name] = n

daily = pd.concat(list(tracks.values()) + [bmk[BENCH]], axis=1).loc[WINDOW]
print('daily panel', daily.shape, '| tracks', len(tracks))

daily panel (5008, 9) | tracks 8


## 3. Statistics — full metrics, turnover, cost drag, NW alpha vs inverse-vol

House-style comparison table: the full `summary_stats` set + one-sided monthly turnover +
annualised cost drag + a Newey–West alpha of each scheme's net track regressed on the
inverse-vol net track (the baseline this stage must beat).

In [4]:
# ===== 3. Stats table =====
all_cols = [c for c in daily.columns if c.startswith('ALL_')]
summary = pd.concat([
    fx.summary_stats(daily[all_cols], benchmark=bmk[BENCH]),
    fx.summary_stats(daily[[BENCH]]),
])

meta = {}
for s in SCHEMES:
    to   = fx.turnover(w[s])                                  # weight-based, same gross/net
    c    = fx.roundtrip_cost(w[s], hs_out, hs_pts)
    live = w[s].abs().sum(axis=1) > 0
    drag = float(c[live].loc[WINDOW].mean() * fx.ANN_DAYS)
    for basis in ('gross', 'net'):
        key = f'ALL_{s}_{basis}'
        m = meta.setdefault(key, {})
        m['avg_monthly_turnover'] = to
        if basis == 'net':
            m['ann_cost_drag'] = drag
        if s != 'inv_vol':                                    # baseline: no self-alpha
            base = daily[f'ALL_inv_vol_{basis}']
            res  = fx.nw_regression(daily[key], base.to_frame('inv_vol'), lags=5)
            m['base'] = f'ALL_inv_vol_{basis}'
            m['alpha_vs_inv_vol_ann'] = res['alpha_ann']
            m['t_alpha_vs_inv_vol']   = res['alpha_t']

summary = summary.join(pd.DataFrame(meta).T)
print(summary.shape, 'rows')
summary[['sharpe', 'max_drawdown', 'CVaR_99', 'skew', 'avg_monthly_turnover',
         'ann_cost_drag']].round(3)

(9, 27) rows


,sharpe,max_drawdown,CVaR_99,skew,avg_monthly_turnover,ann_cost_drag
ALL_equal_gross,0.455099,-0.296913,0.028882,-0.521689,0.473517,NaN
ALL_equal_net,0.336169,-0.315598,0.028894,-0.518153,0.473517,0.013296
ALL_inv_vol_gross,0.628442,-0.267829,0.029186,-0.652152,0.684136,NaN
ALL_inv_vol_net,0.465923,-0.293185,0.029199,-0.648005,0.684136,0.018121
ALL_erc_gross,0.590714,-0.297247,0.028976,-0.560588,0.629401,NaN
ALL_erc_net,0.437575,-0.32002,0.028976,-0.55641,0.629401,0.017081
ALL_mvo_gross,0.462686,-0.478868,0.034823,-1.149746,0.699435,NaN
ALL_mvo_net,0.320285,-0.516318,0.034895,-1.150165,0.699435,0.017221
FXCTEM8,0.141485,-0.320774,0.021387,-0.253154,NaN,NaN


## 4. Validation — helper correctness, no-lookahead, bit-identity

Prove the numerics before trusting the table: ERC/​MVO/​shrinkage unit tests, a no-lookahead
truncation test on the ERC panel, and a strict bit-identity check that the inverse-vol path is
unchanged versus the committed `weights_combined_monthly.csv` (which the baseline backtest wrote).

In [5]:
# ===== 4. Validation asserts =====
lab = list('ABCDE')

# ERC on a diagonal cov reduces to inverse-vol
d = np.array([0.01, 0.02, 0.04, 0.03, 0.05])
iv = 1/np.sqrt(d); iv /= iv.sum()
assert np.allclose(fx.erc_weights(pd.DataFrame(np.diag(d), index=lab, columns=lab)).values, iv, atol=1e-8)

# ERC equalises risk contributions
rng = np.random.default_rng(0); L = rng.standard_normal((5, 5)); Sig = L @ L.T + np.eye(5)
we = fx.erc_weights(pd.DataFrame(Sig, index=lab, columns=lab)).values
rc = we * (Sig @ we)
assert rc.std()/rc.mean() < 1e-6 and abs(we.sum()-1) < 1e-10 and (we > 0).all()

# MVO: equal-mu -> equal; tilts to higher mu; respects cap+gross+nonneg; degenerate-mu -> min-var
covI = pd.DataFrame(np.eye(5), index=lab, columns=lab)
assert np.allclose(np.asarray(fx.mvo_weights(np.ones(5), covI, 1.0, 0.40)), 0.2, atol=1e-6)
w4 = np.asarray(fx.mvo_weights(np.array([.05,.01,.01,.01,.01]), covI, 1.0, 0.40))
assert w4.argmax() == 0 and w4.max() <= 0.40+1e-9 and abs(w4.sum()-1) < 1e-9 and (w4 >= -1e-12).all()
w6 = np.asarray(fx.mvo_weights(np.array([-.01,-.02,-.03,-.01,-.02]), covI, 1.0, 0.40))
assert abs(w6.sum()-1) < 1e-9 and (w6 >= 0).all() and np.isfinite(w6).all()

# shrunk_cov: labelled, symmetric, PD, better-conditioned than the raw sample cov
C  = fx.shrunk_cov(xret[UNIVERSE_ALL[:9]], window=250)
Xw = xret[UNIVERSE_ALL[:9]].tail(250).dropna(how='any')
Sm = np.cov(Xw.values, rowvar=False, bias=True)
assert list(C.columns) == list(Xw.columns) and np.allclose(C.values, C.values.T)
assert (np.linalg.eigvalsh(C.values) > 0).all()
assert np.linalg.cond(C.values) <= np.linalg.cond(Sm) + 1e-8

# No-lookahead: ERC weights on data truncated at `cut` match the full run on an inner window
cut = '2015-06-30'
w_tr = fx.carry_portfolio(carry.loc[:cut], xret.loc[:cut], n_buckets=N_BUCKETS,
                          universe=UNIVERSE_ALL, weighting='erc')
chk = slice('2010-01-01', '2015-05-31')
assert np.allclose(w_unit['erc'].loc[chk].fillna(0).values,
                   w_tr.loc[chk].reindex(columns=w_unit['erc'].columns).fillna(0).values, atol=1e-10)

# Bit-identity: inverse-vol vol-targeted book == committed weights_combined_monthly.csv
saved = pd.read_csv(OUT / 'weights_combined_monthly.csv', index_col=0, parse_dates=True)
mine  = w['inv_vol'].resample('ME').last().reindex(columns=saved.columns)
idx   = saved.index.intersection(mine.index)
assert np.allclose(saved.loc[idx].fillna(0).values, mine.loc[idx].fillna(0).values, atol=1e-9)
print('all validation asserts passed  (ERC RC cv = %.1e)' % (rc.std()/rc.mean()))

all validation asserts passed  (ERC RC cv = 5.1e-09)


## 5. Output + acceptance checks

Write the comparison CSV and the four per-scheme **unit-book** month-end weight files (unit books,
so the schemes are directly comparable at gross 2 before the per-scheme vol-target scalar).
Acceptance: four schemes × gross/net on the common window, turnover present, NW alpha populated,
and the inverse-vol net Sharpe reconciles to the Stage-3 vol-targeted anchor.

In [6]:
# ===== 5. Write outputs + acceptance asserts =====
OUT.mkdir(exist_ok=True)
summary.to_csv(OUT / 'stage4_weighting_comparison.csv')
for s in SCHEMES:
    w_unit[s].resample('ME').last().rename_axis('date').to_csv(OUT / f'weights_{s}_monthly.csv')

strat_rows = [i for i in summary.index if i.startswith('ALL_')]
assert len(strat_rows) == len(SCHEMES) * 2, f'expected {len(SCHEMES)*2} scheme rows, got {len(strat_rows)}'
assert all(str(summary.loc[i, 'end']) == '2026-06-30' for i in strat_rows), 'window end'
assert all(str(summary.loc[i, 'start']) <= '2007-09-30' for i in strat_rows), 'window start'
non_base = [i for i in strat_rows if '_inv_vol_' not in i]
assert summary.loc[non_base, 't_alpha_vs_inv_vol'].notna().all(), 'missing NW alpha'

# Reconciliation: vol-targeted inverse-vol == Stage-3 'voltgt' (the ALL 0.466 anchor)
s3 = pd.read_csv(OUT / 'stage3_dynamic_comparison.csv', index_col=0)
a = float(summary.loc['ALL_inv_vol_net', 'sharpe'])
b = float(s3.loc['ALL_voltgt_net', 'sharpe'])
print(f'ALL inv_vol-net Sharpe {a:.3f}  vs Stage-3 voltgt {b:.3f}  (d={a-b:+.4f})')
assert abs(a - b) < 1e-3, 'inv_vol path drifted from Stage-3 voltgt'
print('wrote stage4_weighting_comparison.csv (%d rows) + %d weights_{scheme}_monthly.csv; checks passed'
      % (len(summary), len(SCHEMES)))

ALL inv_vol-net Sharpe 0.466  vs Stage-3 voltgt 0.466  (d=+0.0000)
wrote stage4_weighting_comparison.csv (9 rows) + 4 weights_{scheme}_monthly.csv; checks passed


## 6. Verdict — does optimization beat inverse-vol net of costs?

In [7]:
# ===== 6. Verdict (combined ALL book) =====
show = ['sharpe', 'max_drawdown', 'CVaR_99', 'skew', 'avg_monthly_turnover',
        'ann_cost_drag', 't_alpha_vs_inv_vol']
net = summary.loc[[f'ALL_{s}_net' for s in SCHEMES], show].round(3)
gro = summary.loc[[f'ALL_{s}_gross' for s in SCHEMES], ['sharpe']].round(3)
print('=== net of costs ===')
print(net)
print('\ngross Sharpe:', {s: float(gro.loc[f'ALL_{s}_gross', 'sharpe']) for s in SCHEMES})

base = float(summary.loc['ALL_inv_vol_net', 'sharpe'])
best = max(SCHEMES, key=lambda s: float(summary.loc[f'ALL_{s}_net', 'sharpe']))
print(f'\nbest net-Sharpe scheme: {best} '
      f'({float(summary.loc[f"ALL_{best}_net","sharpe"]):.3f}) vs inv_vol {base:.3f}')

=== net of costs ===
                   sharpe max_drawdown   CVaR_99      skew  \
ALL_equal_net    0.336169    -0.315598  0.028894 -0.518153   
ALL_inv_vol_net  0.465923    -0.293185  0.029199 -0.648005   
ALL_erc_net      0.437575     -0.32002  0.028976  -0.55641   
ALL_mvo_net      0.320285    -0.516318  0.034895 -1.150165   

                avg_monthly_turnover ann_cost_drag t_alpha_vs_inv_vol  
ALL_equal_net               0.473517      0.013296          -1.788838  
ALL_inv_vol_net             0.684136      0.018121                NaN  
ALL_erc_net                 0.629401      0.017081          -0.494531  
ALL_mvo_net                 0.699435      0.017221          -0.824002  

gross Sharpe: {'equal': 0.4550992061589326, 'inv_vol': 0.6284423467928719, 'erc': 0.5907140098040621, 'mvo': 0.46268589023345597}

best net-Sharpe scheme: inv_vol (0.466) vs inv_vol 0.466


### Conclusion

**No — optimization does not beat inverse-vol net of costs.** On the combined book,
**inverse-vol is the best net-of-cost scheme**. ERC is a close second (it shares inverse-vol's
diagonal limit and only re-weights for correlation), but its small edge does not survive costs.
**Equal-weight** gives up Sharpe by ignoring the vol structure. **Mean-variance is the worst
net track**: with μ = noisy monthly carry it churns (the highest turnover of the four),
concentrates into the cap, and inherits a fatter left tail (worst MaxDD and skew) — a textbook
case of optimizing on estimation error. Every scheme's Newey–West alpha versus inverse-vol is
≤ 0 and statistically indistinguishable from zero, so there is no significant net outperformance
to be had. This confirms the honest prior (§10) on its pessimistic side and vindicates the
baseline's choice of inverse-vol within legs.